# Character Personality Agent 테스트 (Production Level)

캐릭터 성격 특성 추출 에이전트 테스트 노트북

## 역할: "Personality Analyst" (성격 분석가)
- 핵심 성격 (core_traits) - **영구적 특성**
- 결함 (flaws) - **영구적 결함**
- 가치관 (values)

## AI 행동 제어 필드
- **decision_style**: 의사결정 스타일 (rationality, risk_tolerance, biases)
- **stress_response**: 스트레스 반응 (trigger, response_type, breaking_point)
- **social_orientation**: 관계 성향 (trust_default, empathy_level, authority_response)

## 📖 문학적 분석 필드 ✨ NEW
- **character_arc**: 캐릭터 아크 (potential_growth, fatal_flaw, arc_direction)
- **internal_monologue**: 내면 독백 스타일 (thought_process, inner_voice_tone)
- **complex_emotions**: 복합 감정 (동경+질투, 사랑+부담 등)

> **⚠️ 핵심 검증**: `두려움`이 flaws에 있으면 **FAIL**

In [1]:
import sys, os, json, asyncio
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path: sys.path.insert(0, project_root)
from dotenv import load_dotenv
load_dotenv(os.path.join(project_root, '.env'))
print(f"Project root: {project_root}")

Project root: c:\jungle\weapon\sto-link-AI-backend


In [2]:
SAMPLE_STORY = """아린은 어두운 숲 한가운데 서 있었다. 스물다섯 살의 젊은 여전사는 긴 검은 머리카락을 바람에 휘날리며, 손에 쥔 은빛 검을 꼭 움켜쥐었다. 그녀의 눈은 날카롭고 경계심이 가득했다. '누구야?' 아린이 외쳤다. 그녀의 목소리는 단호했지만 약간의 두려움이 섞여 있었다.

두 사람은 한때 '은빛 여명' 기사단의 동맹이었다. 하지만 5년 전 대전쟁 이후, 카엘은 왕국을 배신하고 '암흑회'에 가담했다.

아린은 입술을 깨물며 전투 자세를 취했다. '배신자와 할 말은 없어.' 그녀의 심장이 빠르게 뛰기 시작했다."""

def create_base_state(story=SAMPLE_STORY):
    return {"content": story, "completed_agents": [], "errors": [], "messages": []}

def run_async(coro):
    try:
        loop = asyncio.get_event_loop()
        if loop.is_running():
            import nest_asyncio; nest_asyncio.apply()
            return loop.run_until_complete(coro)
        return asyncio.run(coro)
    except: return asyncio.run(coro)

## 1. Personality Agent 실행

In [3]:
from app.agents.extraction.character.personality import personality_extraction_node

async def test_personality():
    print("🧠 Personality Agent 테스트...")
    return await personality_extraction_node(create_base_state())

result = run_async(test_personality())

if result.get('errors'):
    print(f"\n❌ 에러 발생:")
    for err in result.get('errors', []):
        print(f"   {err}")
else:
    personality_data = result.get('char_personality', {})
    print(f"\n✅ 추출 완료:")
    print(f"   - 캐릭터 수: {len(personality_data)}개")
    print(f"   - 이름: {list(personality_data.keys())}")

🧠 Personality Agent 테스트...

✅ 추출 완료:
   - 캐릭터 수: 2개
   - 이름: ['아린', '카엘']


## 2. 기본 성격 출력

In [4]:
personality_data = result.get('char_personality', {})

if not personality_data:
    print("❌ 캐릭터 데이터 없음")
else:
    print("="*70)
    print("🧠 기본 성격 (core_traits, flaws, values)")
    print("="*70)

    for name, data in personality_data.items():
        print(f"\n🧑 {name}")
        print(f"   핵심 성격: {data.get('core_traits', [])}")
        print(f"   결함: {data.get('flaws', [])}")
        print(f"   가치관: {data.get('values', [])}")

🧠 기본 성격 (core_traits, flaws, values)

🧑 아린
   핵심 성격: ['용기 있는', '경계심 많은']
   결함: ['자기 의존적인', '타인에 대한 불신']
   가치관: ['정의', '충성심']

🧑 카엘
   핵심 성격: ['냉소적인', '계산적인']
   결함: ['극단적 이상주의', '환멸']
   가치관: ['명예', '권력']


## 3. 📈 Character Arc (캐릭터 아크) - NEW

In [5]:
print("="*70)
print("📈 Character Arc (캐릭터 아크)")
print("="*70)

personality_data = result.get('char_personality', {})

for name, data in personality_data.items():
    arc = data.get('character_arc', {})
    if arc:
        print(f"\n🧑 {name}")
        print(f"   🌱 성장 가능성: {arc.get('potential_growth', 'N/A')}")
        print(f"   💀 치명적 결점: {arc.get('fatal_flaw', 'N/A')}")
        print(f"   📊 아크 방향: {arc.get('arc_direction', 'N/A')}")
        print(f"   ⚔️ 내적 갈등: {arc.get('internal_conflict', 'N/A')}")
    else:
        print(f"\n🧑 {name}: character_arc 없음")

📈 Character Arc (캐릭터 아크)

🧑 아린
   🌱 성장 가능성: 배신으로 상처받은 마음을 치유하고 다시 신뢰하는 법을 배움
   💀 치명적 결점: 지나친 자기 의존 또는 타인에 대한 불신
   📊 아크 방향: Growth
   ⚔️ 내적 갈등: 과거의 동료에 대한 신뢰 vs 배신에 대한 분노

🧑 카엘
   🌱 성장 가능성: 자신의 배신 이유를 직면하고 속죄의 길을 찾음
   💀 치명적 결점: 극단적 이상주의 또는 환멸로 인한 냉소
   📊 아크 방향: Fall
   ⚔️ 내적 갈등: 과거의 명예 vs 현재의 선택을 정당화하려는 욕구


## 4. 🗣️ Internal Monologue (내면 독백) - NEW

In [6]:
print("="*70)
print("🗣️ Internal Monologue (내면 독백 스타일)")
print("="*70)

personality_data = result.get('char_personality', {})

for name, data in personality_data.items():
    mono = data.get('internal_monologue', {})
    if mono:
        print(f"\n🧑 {name}")
        print(f"   🧠 사고 방식: {mono.get('thought_process', 'N/A')}")
        print(f"   🎭 내면 목소리 톤: {mono.get('inner_voice_tone', 'N/A')}")
        print(f"   🔄 반복되는 생각: {mono.get('recurring_thoughts', [])}")
    else:
        print(f"\n🧑 {name}: internal_monologue 없음")

🗣️ Internal Monologue (내면 독백 스타일)

🧑 아린
   🧠 사고 방식: Intuitive-first
   🎭 내면 목소리 톤: Self-critical
   🔄 반복되는 생각: ['배신에 대한 분노', '동료에 대한 실망']

🧑 카엘
   🧠 사고 방식: Analytical
   🎭 내면 목소리 톤: Philosophical
   🔄 반복되는 생각: ['과거의 영광', '현재의 선택을 정당화하기']


## 5. 💔 Complex Emotions (복합 감정) - NEW

In [7]:
print("="*70)
print("💔 Complex Emotions (복합 감정)")
print("="*70)

personality_data = result.get('char_personality', {})

for name, data in personality_data.items():
    emotions = data.get('complex_emotions', [])
    print(f"\n🧑 {name}")
    if emotions:
        for i, emotion in enumerate(emotions, 1):
            print(f"   {i}. {emotion}")
    else:
        print("   (복합 감정 없음)")

💔 Complex Emotions (복합 감정)

🧑 아린
   1. 동료에 대한 실망감
   2. 배신자에 대한 분노

🧑 카엘
   1. 과거 동료에 대한 미련
   2. 현재 선택에 대한 자기 합리화


## 6. AI 행동 제어 필드

In [8]:
print("="*70)
print("🤖 AI 행동 제어 필드 (decision_style, stress_response, social_orientation)")
print("="*70)

personality_data = result.get('char_personality', {})

for name, data in personality_data.items():
    print(f"\n🧑 {name}")
    
    # Decision Style
    ds = data.get('decision_style', {})
    if ds:
        r = ds.get('rationality', 0.5)
        r_bar = '█' * int(r * 10) + '░' * (10 - int(r * 10))
        print(f"   Rationality: [{r_bar}] {r:.1f}")
        rt = ds.get('risk_tolerance', 0.5)
        rt_bar = '█' * int(rt * 10) + '░' * (10 - int(rt * 10))
        print(f"   Risk Tolerance: [{rt_bar}] {rt:.1f}")
    
    # Stress Response
    sr = data.get('stress_response', {})
    if sr and sr.get('trigger'):
        print(f"   Stress Trigger: {sr.get('trigger')}")
    
    # Social Orientation
    so = data.get('social_orientation', {})
    if so:
        print(f"   Trust Default: {so.get('trust_default', 0):+d}")

🤖 AI 행동 제어 필드 (decision_style, stress_response, social_orientation)

🧑 아린
   Rationality: [██████░░░░] 0.6
   Risk Tolerance: [███████░░░] 0.7
   Stress Trigger: 배신
   Trust Default: -2

🧑 카엘
   Rationality: [████████░░] 0.8
   Risk Tolerance: [██████░░░░] 0.6
   Stress Trigger: 과거의 명예 상실
   Trust Default: -3


## 7. Full JSON 출력

In [9]:
print("="*70)
print("📄 Full JSON Output")
print("="*70)
personality_data = result.get('char_personality', {})
if personality_data:
    print(json.dumps(personality_data, ensure_ascii=False, indent=2))
else:
    print("{}")

📄 Full JSON Output
{
  "아린": {
    "name": "아린",
    "core_traits": [
      "용기 있는",
      "경계심 많은"
    ],
    "flaws": [
      "자기 의존적인",
      "타인에 대한 불신"
    ],
    "values": [
      "정의",
      "충성심"
    ],
    "decision_style": {
      "rationality": 0.6,
      "risk_tolerance": 0.7,
      "biases": [
        "권위 편향"
      ]
    },
    "stress_response": {
      "trigger": "배신",
      "response_type": "경계심 증가",
      "breaking_point": "완전한 절망감"
    },
    "social_orientation": {
      "trust_default": -2,
      "empathy_level": 3,
      "authority_response": "선별적 복종"
    },
    "character_arc": {
      "potential_growth": "배신으로 상처받은 마음을 치유하고 다시 신뢰하는 법을 배움",
      "fatal_flaw": "지나친 자기 의존 또는 타인에 대한 불신",
      "arc_direction": "Growth",
      "internal_conflict": "과거의 동료에 대한 신뢰 vs 배신에 대한 분노"
    },
    "internal_monologue": {
      "thought_process": "Intuitive-first",
      "inner_voice_tone": "Self-critical",
      "recurring_thoughts": [
        "배신에 대한 분노",
        "동료에 대한 실망"
 

## 8. Production 체크리스트

In [10]:
print("="*70)
print("✅ Production 체크리스트")
print("="*70)

personality_data = result.get('char_personality', {})
checks = []

if len(personality_data) >= 1:
    checks.append(("✅", f"{len(personality_data)} characters extracted"))
else:
    checks.append(("❌", "No characters"))

if personality_data:
    # Core traits
    has_traits = any(data.get('core_traits') for data in personality_data.values())
    checks.append(("✅" if has_traits else "⚠️", "Core traits"))
    
    # Character Arc
    has_arc = any(data.get('character_arc', {}).get('arc_direction') for data in personality_data.values())
    checks.append(("✅" if has_arc else "⚠️", "Character Arc"))
    
    # Internal Monologue
    has_mono = any(data.get('internal_monologue', {}).get('thought_process') for data in personality_data.values())
    checks.append(("✅" if has_mono else "⚠️", "Internal Monologue"))
    
    # Complex Emotions
    has_complex = any(data.get('complex_emotions') for data in personality_data.values())
    checks.append(("✅" if has_complex else "⚠️", "Complex Emotions"))
    
    # Decision Style
    has_decision = any(data.get('decision_style') for data in personality_data.values())
    checks.append(("✅" if has_decision else "⚠️", "Decision Style"))
    
    # Emotion separation
    TEMPORARY = ['두려움', 'fear', '불안', 'anxious']
    sep_ok = True
    for data in personality_data.values():
        for flaw in data.get('flaws', []):
            if any(e in str(flaw).lower() for e in TEMPORARY):
                sep_ok = False
    checks.append(("✅" if sep_ok else "❌", "Emotion separation"))

print()
for status, msg in checks:
    print(f"{status} {msg}")

print("\n" + "=" * 70)
passed = sum(1 for s, _ in checks if s == "✅")
print(f"결과: {passed}/{len(checks)} checks passed")

✅ Production 체크리스트

✅ 2 characters extracted
✅ Core traits
✅ Character Arc
✅ Internal Monologue
✅ Complex Emotions
✅ Decision Style
✅ Emotion separation

결과: 7/7 checks passed


## 9. 디버그 정보

In [11]:
print("="*70)
print("🔍 디버그 정보")
print("="*70)
print(f"Result keys: {result.keys()}")
print(f"Errors: {result.get('errors', [])}")
print(f"Completed agents: {result.get('completed_agents', [])}")

🔍 디버그 정보
Result keys: dict_keys(['char_personality', 'completed_agents', 'messages'])
Errors: []
Completed agents: ['personality']
